In [1]:
import os
import json

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

In [2]:
# base_path = "../../../db/object-detection"
# base_path = "/mnt/pg-fs01/Current/DB/ObjectDetectionV3"
base_path = "../../../data/ObjectDetectionV3"
df = pd.read_csv(os.path.join(base_path, "annotation.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))
df_imgs.head()

,id,name,width,height,batch,folder,group_id
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Cam3_H243_discount_14.png,640.0,480.0,batch1_10.5,batch1,5204194a-5097-42a7-b81a-94791a1e482a
1,fc377a4c-4a15-444d-85e7-ce8a3a578a8e,Cam3_H243_discount_15.png,640.0,480.0,batch1_10.5,batch1,5204194a-5097-42a7-b81a-94791a1e482a
2,ff50bde4-3825-47b8-9cab-cc97663f1c97,Cam3_H243_discount_16.png,640.0,480.0,batch1_10.5,batch1,5204194a-5097-42a7-b81a-94791a1e482a
3,5cec4eb5-edd9-4831-9ca3-5cfb04fc6d82,Cam3_H243_discount_17.png,640.0,480.0,batch1_10.5,batch1,5204194a-5097-42a7-b81a-94791a1e482a
4,b92da22b-21df-406f-8a0b-3c3336d8393a,Cam3_H243_discount_18.png,640.0,480.0,batch1_10.5,batch1,5204194a-5097-42a7-b81a-94791a1e482a


In [3]:
print("before", df_imgs.shape, df.shape)

# TODO: do not remove image without annotations when problem with batch4_22.5/5 will be fixed
# rm images without annotations
tmp_df = pd.merge(df, df_imgs, how="outer", left_on="img_id", right_on="id")
tmp_ids = tmp_df[tmp_df["id_x"].isna()]["id_y"].values
df_imgs = df_imgs[~df_imgs["id"].isin(tmp_ids)]
print("number of images without annotations:", len(tmp_ids))

# rm images with problems in annotations
tmp_ids = df[(df["xtl"].isna()) | (df["x"].isna())]["img_id"].unique()
df_imgs = df_imgs[~df_imgs["id"].isin(tmp_ids)]
df = df[~df["img_id"].isin(tmp_ids)]
print("number of images with 'problematic' annotations:", len(tmp_ids))

#rm duplicates
df1 = df_imgs.groupby(['name','batch', "folder", "id"]).size().reset_index()[['name','batch', "folder", "id"]]
duplicates = df1[df1.duplicated('name', keep=False)]
tmp_ids = duplicates.iloc[1::2]["id"].values
df_imgs = df_imgs[~df_imgs["id"].isin(tmp_ids)]
df = df[~df["img_id"].isin(tmp_ids)]
print("number of duplicates:", len(tmp_ids))

print("after", df_imgs.shape, df.shape)

before (1457, 7) (35977, 15)
number of images without annotations: 71
number of images with 'problematic' annotations: 57
number of duplicates: 10
after (1319, 7) (34476, 15)


In [5]:
image_ids = df_imgs["id"].values
group_ids = df_imgs["group_id"].values
labels = sorted(df["label"].unique())

val_size: float = 0.1
randon_seed: int = 11
num_skip_valid_splits: int = 0

n_splits = round(1 / val_size)
gskf = GroupShuffleSplit(n_splits=n_splits, random_state=randon_seed)
current_try = 0
# Add check for classes and groups
for index, indices in enumerate(gskf.split(image_ids, groups=group_ids)):
    print(f"Trying split # {index}")
    train_indexes, test_indexes = indices
    train_ids = image_ids[train_indexes]
    test_ids = image_ids[test_indexes]
    train_classes = sorted(df[df["img_id"].isin(train_ids)]["label"].unique())
    test_classes = sorted(df[df["img_id"].isin(test_ids)]["label"].unique())

    is_satisfactory_split = labels == train_classes and labels == test_classes
    if is_satisfactory_split:
        if current_try >= num_skip_valid_splits:
            print("OK")
            break
        else:
            current_try += 1
    elif index == n_splits-1:
        print("not OK")
        break
    
print(f"train: {len(train_ids)} test: {len(test_ids)}")

with open(os.path.join(base_path, "train_test_split.json"), "w") as f:
    json.dump({"train": train_ids.tolist(), "test": test_ids.tolist()}, f)

Trying split # 0
Trying split # 1
Trying split # 2
OK
train: 1027 test: 292
